# Training data chip creation
This notebook shows you how the Pipeline can be used to create 300x300 training chips for the toy model. These chips are matched with training COCO labels, and passed through the model at training time.

## Purpose of this notebook
This notebook is designed to get you familiar with the chip making workflow; 
- Load all COCO training image, label pairs
- For each training sample: 
    - Create a tiling Pipeline object
    - Query Pipeline by bounding box in lat, lon coords (upper left and lower right coordinates) and product ID (saves only matching PIDs)
    - Match images across LTM tile indices (1-4 tiles per query)
    - Merge, reproject, clip to training AOI, save to file
    - Copy over matching label
    - Visualize results
- Clean up memory

# Setup
Imports and repo clone.

In [ ]:
import os
# Configure environment variable to suppress warning
os.environ['MALLOC_CONF'] = 'oversize_threshold:1,background_thread:true,metadata_thp:auto'

import json
import logging
from glob import glob
from contextlib import redirect_stdout, nullcontext
from functools import partial
from collections import Counter
from io import StringIO
import multiprocessing as mp
from multiprocessing import Pool, cpu_count, get_logger
import warnings

import rasterio
import xarray as xr
from rasterio.enums import Resampling
from rasterio.crs import CRS
from tqdm import tqdm
import subprocess
from pathlib import Path
import sys
import geopandas as gpd
import time
import rioxarray as rxr
import shutil
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from getpass import getuser

In [ ]:
# Get the repo root directory (parent of notebooks/)
repo_root = Path.cwd().parent.parent

# Convert /panfs path to /explore path (JupyterHub quirk)
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)

# Verify we're in the right location
if not (repo_root / "model").exists():
    raise FileNotFoundError(
        "Cannot find lfm/model directory. "
        "Please ensure you're running this notebook from the lfm/notebooks/ directory."
    )

# Add the parent of the repo to sys.path for imports
sys.path.insert(0, str(repo_root.parent))

# Import required modules
from lfm.model.Pipeline import Pipeline
from lfm.model.chip_making.chip_utils import (
    extract_product_id, run_pipeline_for_sample, get_worker_logger, group_cubes_by_tile,
    merge_and_reproject_datasets, clip_and_combine_datasets, write_chip_to_tif
)
from lfm.model.chip_making.chip_constants import (
    PROJECT_DIR, GPKG_PATH, TILE_DB_PATH, ZOOM_LEVEL, CHIP_DIR, LABEL_DIR, COMMON_NODATA
)
from lfm.lfm.toy_model.all_tasks.all_utils import prepare_output_dir
print("✓ Successfully imported LFM modules")

In [ ]:
logger = get_worker_logger()

# User configuration
`DELETE_PREV_OUTPUTS`: whether to delete previous outputs; this is advised to be True, but is False by default.

`OUTPUT_DIR`: base output dir where you want the tiling outputs to go. 

<mark>Note: the notebook will create a subdirectory of this for your user; for Sandy, it creates one called "ajkerr1". If you want to do multiple experiments, make sure to change the directory path so that your outputs aren't overwritten!</mark>

`STATIC_BANDS`: exact band names of static bands to use in chip creation. If you would like to omit some of the bands, you can comment them out by hightlighting them and typing `#`.

In [ ]:
DELETE_PREV_OUTPUTS = True  # Whether to delete previous chip creation runs

OUTPUT_DIR = Path("/explore/nobackup/people/ajkerr1/Lunar_FM/nac_test")
OUTPUT_DIR.mkdir(exist_ok=True)

STATIC_BANDS = [
    # "LDRM_32_N_FLOAT.iau",
    # "GlobeNoPolesDeltaCPR_v2-offsetto49d.iau",
    # "GlobeNoPolesDeltaS1_v2.iau",
    # "WAC_EMP_321NM.iau",
    # "WAC_EMP_360NM.iau",
    # "WAC_EMP_415NM.iau",
    # "WAC_EMP_566NM.iau",
    # "WAC_EMP_604NM.iau",
    # "WAC_EMP_643NM.iau",
    # "WAC_EMP_689NM.iau",
    # "WAC_GLOBAL.iau",
    # "WAC_TIO2.iau",
    # "jggrx_1800f_me_dist_meters_20km_cog",
    # "hpar_global128ppd_v1c_dateline_cut.iau3",
    # "RA_SAM_70Sto70N.iau7",
    # "diviner_tbol_snapshot_000E",
    # "diviner_tbol_snapshot_015E",
    # "diviner_tbol_snapshot_030E",
    # "diviner_tbol_snapshot_045E",
    # "diviner_tbol_snapshot_060E",
    # "diviner_tbol_snapshot_075E",
    # "diviner_tbol_snapshot_090E",
    # "diviner_tbol_snapshot_105E",
    # "diviner_tbol_snapshot_120E",
    # "diviner_tbol_snapshot_135E",
    # "diviner_tbol_snapshot_150E",
    # "diviner_tbol_snapshot_165E",
    # "diviner_tbol_snapshot_180E",
    # "diviner_tbol_snapshot_195E",
    # "diviner_tbol_snapshot_210E",
    # "diviner_tbol_snapshot_225E",
    # "diviner_tbol_snapshot_240E",
    # "diviner_tbol_snapshot_255E",
    # "diviner_tbol_snapshot_270E",
    # "diviner_tbol_snapshot_285E",
    # "diviner_tbol_snapshot_300E",
    # "diviner_tbol_snapshot_315E",
    # "diviner_tbol_snapshot_330E",
    # "diviner_tbol_snapshot_345E",
    # "TREG_ANOM_70Sto70N.iau7",
    # "Lunar_Kaguya_MIMap_MineralDeconv_ClinopyroxenePercent_50N50S.iau2",
    # "Lunar_Kaguya_MIMap_MineralDeconv_FeOWeightPercent_50N50S.iau2",
    # "Lunar_Kaguya_MIMap_MineralDeconv_OlivinePercent_50N50S.iau2",
    # "Lunar_Kaguya_MIMap_MineralDeconv_OpticalMaturityIndex_50N50S.iau2",
    # "Lunar_Kaguya_MIMap_MineralDeconv_OrthopyroxenePercent_50N50S.iau2",
    # "Lunar_Kaguya_MIMap_MineralDeconv_PlagioclaseGrainSizeMicrons_50N50S.iau2",
    # "Lunar_Kaguya_MIMap_MineralDeconv_PlagioclasePercent_50N50S.iau2",
    # "kaguya_mi_derived_30ppd_mpfe.iau",
    # "kaguya_mi_derived_30ppd_npfe.iau",
    # "kaguya_mi_derived_30ppd_smfe.iau",
    # "Lunar_Kaguya_MIMap_Band1_MV1_414nm_65N65S_512ppd.iau2",
    # "Lunar_Kaguya_MIMap_Band2_MV2_749nm_65N65S_512ppd.iau2",
    # "Lunar_Kaguya_MIMap_Band3_MV3_901nm_65N65S_512ppd.iau2",
    # "Lunar_Kaguya_MIMap_Band4_MV4_950nm_65N65S_512ppd.iau2",
    # "Lunar_Kaguya_MIMap_Band5_MV5_1001nm_65N65S_512ppd.iau2",
    # "Lunar_Kaguya_MIMap_Band7_MN2_1049nm_65N65S_512ppd.iau2",
    # "Lunar_Kaguya_MIMap_Band8_MN3_1248nm_65N65S_512ppd.iau2",
    # "Lunar_Kaguya_MIMap_Band9_MN4_1548nm_65N65S_512ppd.iau2",
    # "lola_kaguya_60mpp_asp",
    # "lola_kaguya_60mpp_cos",
    # "lola_kaguya_60mpp_elv",
    # "lola_kaguya_60mpp_sin",
    # "lola_kaguya_60mpp_slp",
]

In [ ]:
prep = prepare_output_dir(OUTPUT_DIR, DELETE_PREV_OUTPUTS)

## NAC-only chip helper functions

Use this path when you only want LRO NAC imagery. It creates/uses a NAC tile database, runs a dynamic-only Pipeline helper with product-ID filtering, skips static imagery entirely, merges/reprojects the NAC datacubes to the original COCO chip grid, and writes a one-band `*_input_nac_chip.tif`.


In [ ]:
from osgeo import gdal
from rioxarray.merge import merge_arrays

from lfm.model.TmsIntersector import TmsIntersector
from lfm.model.TmsTileDef import TmsTileDef

MOON_SRS = (
    'GEOGCRS["Moon (2015) - Sphere / Ocentric", '
    'DATUM["Moon (2015) - Sphere", '
    'ELLIPSOID["Moon (2015) - Sphere",1737400,0, '
    'LENGTHUNIT["metre",1]]], '
    'PRIMEM["Reference Meridian",0, '
    'ANGLEUNIT["degree",0.0174532925199433]], '
    'CS[ellipsoidal,2], '
    'AXIS["geodetic latitude (Lat)",north, ORDER[1], '
    'ANGLEUNIT["degree",0.0174532925199433]], '
    'AXIS["geodetic longitude (Lon)",east, ORDER[2], '
    'ANGLEUNIT["degree",0.0174532925199433]], '
    'ID["IAU",30100,2015], '
    'REMARK["Source of IAU Coordinate systems: https://doi.org/10.1007/s10569-017-9805-5"]]'
)

NAC_IMAGE_DIR = PROJECT_DIR / "processed_data/Lunar/LRO_NAC_Pho_Sites"
NAC_CHIP_SUFFIX = "_input_nac_chip"


def get_tile_db_path(image_dir, db_name="output_index.shp"):
    image_dir = Path(image_dir)
    if not image_dir.exists() or not image_dir.is_dir():
        raise ValueError(f"A valid image directory is required: {image_dir}")

    tile_db_path = image_dir / db_name
    if tile_db_path.exists():
        return tile_db_path

    tif_paths = sorted(image_dir.glob("*.tif"))
    if not tif_paths:
        raise FileNotFoundError(f"No .tif files found under {image_dir}")

    gdal.TileIndex(str(tile_db_path), [str(path) for path in tif_paths], outputSRS=MOON_SRS)
    tile_db_path.chmod(0o666)
    return tile_db_path


def product_id_from_path(path):
    stem = Path(path).stem
    parts = stem.split("_")
    for part in parts:
        if part.startswith("M") and len(part) >= 10:
            return part
    return extract_product_id(stem)


def run_nac_pipeline_for_sample(
    train_fn,
    product_id,
    geom_bounds,
    datacube_dir,
    nac_image_dir=NAC_IMAGE_DIR,
    logger=logger,
    zoom_level=ZOOM_LEVEL,
    debug=False,
):
    """Run only the dynamic NAC branch of Pipeline, skipping static imagery."""
    ulLon, lrLat, lrLon, ulLat = geom_bounds
    datacube_dir = Path(datacube_dir)
    datacube_dir.mkdir(parents=True, exist_ok=True)

    tile_db_path = get_tile_db_path(nac_image_dir)
    cube_files = []
    status = "success"

    try:
        with redirect_stdout(sys.stderr):
            pipeline = Pipeline(
                tile_db_path,
                datacube_dir,
                debug=debug,
                targetProductID=product_id,
            )
            print(f"Created NAC-only pipeline instance for PID: {product_id}")

            tile_indexes = TmsIntersector().getTids(ulLat, ulLon, lrLat, lrLon, zoom_level)
            print(f"Found {len(tile_indexes)} intersecting NAC tile(s).")

            for idx in tile_indexes:
                tile_x = idx["tileX"]
                tile_y = idx["tileY"]
                zone = idx["zone"]
                tile_zoom = idx["zoomLevel"]

                print(
                    "Processing NAC tile "
                    f"({tile_x}, {tile_y}) / zone {zone} / zoom {tile_zoom}"
                )

                tile_def = TmsTileDef.initFromParams(zone, tile_zoom)
                ulx, uly, lrx, lry = tile_def.getTileBbox(tile_x, tile_y)
                query_ul_lat, query_ul_lon = tile_def.ltmToLatLon(ulx, uly)
                query_lr_lat, query_lr_lon = tile_def.ltmToLatLon(lrx, lry)

                layer = pipeline._query(query_ul_lat, query_ul_lon, query_lr_lat, query_lr_lon)
                feature_count = layer.GetFeatureCount()
                if feature_count == 0:
                    print("NAC tile does not overlap any images.")
                    continue

                cube = pipeline._createCube(
                    layer,
                    ulx,
                    uly,
                    lrx,
                    lry,
                    tile_def.srs,
                    tile_def.tileWidth,
                    tile_def.tileHeight,
                    is_static=False,
                )
                if len(cube):
                    cube_files.extend(
                        pipeline._writeCube((tile_x, tile_y), cube, tile_def, ulx, uly)
                    )

        if not cube_files:
            logger.warning(f"No NAC datacubes created for {train_fn}")
            status = "warning_no_cubes"

    except Exception as exc:
        logger.error(f"NAC pipeline failed for {train_fn}: {exc}")
        logger.error(f"  Product ID: {product_id}")
        logger.error(f"  Bounds: {geom_bounds}")
        status = "error_pipeline"

    return cube_files, status


def matching_nac_datacubes(cube_files, product_id):
    candidates = [Path(path) for path in cube_files if "Static" not in str(path)]
    matches = [path for path in candidates if f"ProdId-{product_id}" in str(path) or product_id in str(path)]
    if not matches:
        raise FileNotFoundError(
            f"No NAC datacubes matched product_id={product_id}. "
            f"Total non-static datacubes: {len(candidates)}"
        )
    return sorted(matches)


def open_reference_chip(reference_path):
    reference_path = Path(reference_path)
    if not reference_path.exists():
        raise FileNotFoundError(f"Reference chip does not exist: {reference_path}")
    return rxr.open_rasterio(reference_path, masked=True)


def write_nac_chip_from_datacubes(
    nac_files,
    reference_ds,
    output_filename,
    *,
    nodata_value=COMMON_NODATA,
):
    output_filename = Path(output_filename)
    output_filename.parent.mkdir(parents=True, exist_ok=True)

    nac_arrays = [rxr.open_rasterio(path, masked=True) for path in nac_files]
    try:
        merged_nac = merge_arrays(nac_arrays)
        nac_chip = merged_nac.rio.reproject_match(reference_ds)
        if nac_chip.sizes.get("band", 1) > 1:
            nac_chip = nac_chip.isel(band=[0])
        nac_chip = nac_chip.rio.write_nodata(nodata_value, encoded=True)
        nac_chip.rio.to_raster(output_filename)
    finally:
        for array in nac_arrays:
            array.close()
        if "merged_nac" in locals():
            merged_nac.close()
        if "nac_chip" in locals():
            nac_chip.close()

    return output_filename


def copy_matching_label(
    train_fn_no_ext,
    output_dir,
    *,
    label_dir=LABEL_DIR,
    label_suffix="_label",
):
    label_dir = Path(label_dir)
    if not label_dir.exists():
        print(f"Label directory does not exist; skipping label copy: {label_dir}")
        return None

    matches = sorted(label_dir.glob(f"*{train_fn_no_ext}*.npz"))
    if not matches:
        print(f"No matching .npz label found for {train_fn_no_ext}; skipping label copy.")
        return None

    label_output_dir = output_dir / "labels"
    label_output_dir.mkdir(exist_ok=True, parents=True)
    label_output_path = label_output_dir / f"{train_fn_no_ext}{label_suffix}.npz"
    shutil.copy2(matches[0], label_output_path)
    return label_output_path


def load_label_array(label_path):
    label = np.load(label_path, allow_pickle=True)
    if isinstance(label, np.lib.npyio.NpzFile):
        if "data" in label.files:
            return label["data"]
        if "mask" in label.files:
            return label["mask"]
        raise KeyError(f"No supported label key found in {label_path}; keys={label.files}")
    return label


def create_nac_chip_for_sample(
    entry,
    output_dir,
    *,
    product_id=None,
    nac_image_dir=NAC_IMAGE_DIR,
    zoom_level=ZOOM_LEVEL,
    copy_label=True,
):
    train_fn = str(entry["location"])
    geom_bounds = entry["geometry"]  # (ulLon, lrLat, lrLon, ulLat)
    product_id = product_id or product_id_from_path(train_fn)
    train_fn_no_ext = Path(train_fn).stem

    datacube_dir = output_dir / "datacubes" / train_fn_no_ext
    chip_output_dir = output_dir / "chips"
    chip_output_filename = chip_output_dir / f"{train_fn_no_ext}{NAC_CHIP_SUFFIX}.tif"

    cube_files, pipeline_status = run_nac_pipeline_for_sample(
        train_fn=train_fn,
        product_id=product_id,
        geom_bounds=geom_bounds,
        datacube_dir=datacube_dir,
        nac_image_dir=nac_image_dir,
        logger=logger,
        zoom_level=zoom_level,
    )
    if pipeline_status != "success":
        print(f"WARNING: Pipeline status: {pipeline_status}")
    if not cube_files:
        raise RuntimeError("Pipeline produced no NAC datacubes.")

    nac_files = matching_nac_datacubes(cube_files, product_id)
    reference_ds = open_reference_chip(train_fn)
    try:
        chip_path = write_nac_chip_from_datacubes(
            nac_files,
            reference_ds,
            chip_output_filename,
        )
    finally:
        reference_ds.close()

    label_path = copy_matching_label(train_fn_no_ext, output_dir) if copy_label else None

    result = {
        "product_id": product_id,
        "pipeline_status": pipeline_status,
        "num_cube_files": len(cube_files),
        "num_nac_files": len(nac_files),
        "datacube_dir": datacube_dir,
        "chip_path": chip_path,
        "label_path": label_path,
    }
    print(result)
    return result




# Example run: single train sample

### 1. Load previous training geometries into GeoDataFrame
This is our set of AOIs we will use to filter our data with. 

### 2. Run datacube pipeline for single sample
This creates some 512x512 .tifs in the Armstrong tiling scheme. See `tiling_example.ipynb` for examples on how the Pipeline works.

In [ ]:
nac_path = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/PHO")
nac_netcdfs = list(nac_path.glob("*.nc"))
print(f"Found {len(nac_netcdfs)} .nc files.")

In [31]:
import rioxarray as rxr
import xarray as xr
from shapely.geometry import box, mapping
from shapely.ops import transform
from pyproj import CRS, Transformer
import json
from joblib import Parallel, delayed
from tqdm import tqdm

# Define target CRS (IAU Moon geographic)
with open('/panfs/ccds02/nobackup/projects/lfm/IAU_30100_2015.wkt') as f:
    target_wkt = f.read()

def process_netcdf(nc_file: str, target_wkt: str):
    """Process a single NetCDF file and return its spatial metadata."""
    try:
        target_crs = CRS.from_wkt(target_wkt)
        
        ds = xr.open_dataset(nc_file)
        da = ds['band_data']
        da = da.rio.write_crs(da.rio.crs)
        
        source_crs = da.rio.crs
        bounds = da.rio.bounds()
        bbox_geom = box(*bounds)
        
        transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)
        transformed_geom = transform(transformer.transform, bbox_geom)
        transformed_bounds = transformed_geom.bounds
        
        entry = {
            'location': nc_file,
            'geometry': list(transformed_bounds),
        }
        
        ds.close()
        return entry
        
    except Exception as e:
        print(f"\nError processing {nc_file}: {e}")
        return None

# Process files in parallel with joblib
results = Parallel(n_jobs=10, backend='loky', verbose=5)(
    delayed(process_netcdf)(nc_file, target_wkt) 
    for nc_file in tqdm(nac_netcdfs, desc="Processing files")
)

# Filter out None results
entries = [r for r in results if r is not None]
failed_count = len(results) - len(entries)

print(f"\nProcessed {len(entries)} files successfully")
print(f"Failed to process {failed_count} files")

Processing files: 100%|██████████| 766/766 [00:05<00:00, 129.20it/s]



Processed 766 files successfully
Failed to process 0 files


[Parallel(n_jobs=10)]: Done 766 out of 766 | elapsed:    6.2s finished


In [38]:
from lfm.model.Pipeline import Pipeline

tile_db_path = Path("")
pipeline = Pipeline(tile_db_path, Path("."))
for entry in entries:
    if len(pipeline._query(*entry['geometry'])) > 0:
        print(f"Entry {entry['location']} has > 0 overlapping files.")

RuntimeError: Failed to open shapefile: .
  → The file may be corrupted or in an unsupported format

In [ ]:
# self, tileDbPath: Path, outDir: Path, debug: bool = False, targetProductID: str = None

### Run NAC-only chip creation

Run this after defining `entry`. It bypasses the WAC/static cells below and writes a one-band NAC chip.

In [ ]:
train_fn = str(entry["location"])
product_id = "M1127199663L"
train_fn_no_ext = Path(train_fn).stem

print(f"Product ID: {product_id}")
print(f"Reference chip: {train_fn}")

# Save intermediate datacubes under one subdirectory for this sample.
datacube_base_dir = OUTPUT_DIR / "datacubes"
datacube_dir = datacube_base_dir / train_fn_no_ext



In [ ]:
print("Running NAC-only pipeline...")

pipeline_start = time.time()

cube_files, pipeline_status = run_nac_pipeline_for_sample(
    train_fn=train_fn,
    product_id=product_id,
    geom_bounds=entry["geometry"],  # (ulLon, lrLat, lrLon, ulLat)
    datacube_dir=datacube_dir,
    nac_image_dir=NAC_IMAGE_DIR,
    logger=logger,
    zoom_level=ZOOM_LEVEL,
)
pipeline_end = time.time() - pipeline_start

if not cube_files:
    print("WARNING: no NAC datacubes created.")
if pipeline_status != "success":
    print(f"WARNING: Pipeline status: {pipeline_status}")
else:
    print(f"NAC-only pipeline created {len(cube_files)} datacube file(s).")
    print(f"Pipeline ran in {pipeline_end:.2f} seconds.")



### 3. Match NAC datacubes across tile indices

The helper above skipped static imagery. Here we only keep dynamic NAC cubes matching the requested product ID.



In [ ]:
nac_files = matching_nac_datacubes(cube_files, product_id)
print(f"Matched {len(nac_files)} NAC datacube file(s):")
for path in nac_files:
    print(f"  {path}")

### 4. Load reference train chip

This provides the CRS, transform, and pixel grid used to reproject the new NAC chip.



In [ ]:
train_path = Path(train_fn)
reference_ds = open_reference_chip(train_path)

print("Reference chip shape:", reference_ds.shape)
print("Reference CRS:", reference_ds.rio.crs)
print("Reference bounds:", reference_ds.rio.bounds())



### 5. Merge/reproject NAC datacubes to the reference chip grid

This replaces the old WAC/static merge path. The output is a one-band NAC chip aligned to the original COCO chip.



In [ ]:
chip_output_dir = OUTPUT_DIR / "chips"
chip_output_dir.mkdir(exist_ok=True, parents=True)
chip_output_filename = chip_output_dir / f"{train_fn_no_ext}{NAC_CHIP_SUFFIX}.tif"

chip_path = write_nac_chip_from_datacubes(
    nac_files,
    reference_ds,
    chip_output_filename,
)

print(f"Wrote NAC chip: {chip_path}")



In [ ]:
# Inspect the new chip.
new_chip_ds = rxr.open_rasterio(chip_output_filename, masked=True)
print("New NAC chip shape:", new_chip_ds.shape)
print("New NAC chip CRS:", new_chip_ds.rio.crs)
print("New NAC chip bounds:", new_chip_ds.rio.bounds())



### 6. Copy over matching label

This is optional, but useful when making image/label pairs for training.



In [ ]:
label_new_path = copy_matching_label(train_fn_no_ext, OUTPUT_DIR)
print(f"Copied label: {label_new_path}")



### 7. Visualize the NAC chip and label

This compares the newly created NAC chip against the original reference chip and label.



In [ ]:
label_path = next(LABEL_DIR.glob(f"*{train_fn_no_ext}*.npz"), None)
if label_path is None:
    label_path = next(LABEL_DIR.glob(f"*{train_fn_no_ext}*.npy"), None)
if label_path is None:
    print(f"No original label found for {train_fn_no_ext}")
else:
    print(f"Found original label: {label_path}")



# Load chips and labels.
chip_datasets = [rxr.open_rasterio(path, masked=True).values for path in [chip_output_filename, train_path]]
chip_single_bands = [chip.transpose(1, 2, 0)[:, :, 0] for chip in chip_datasets]

label_datasets = []
label_titles = []
if label_new_path is not None:
    label_datasets.append(load_label_array(label_new_path))
    label_titles.append("Copied Label")
if label_path is not None:
    label_datasets.append(load_label_array(label_path))
    label_titles.append("Original Label")

cmap_black_red = ListedColormap(["black", "red"])
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(chip_single_bands[0], cmap="gray")
axes[0, 0].set_title("New NAC Chip")
axes[0, 1].imshow(chip_single_bands[1], cmap="gray")
axes[0, 1].set_title("Reference Chip")

for ax in axes[0, :]:
    ax.axis("off")

for idx, ax in enumerate(axes[1, :]):
    if idx < len(label_datasets):
        ax.imshow(label_datasets[idx], cmap=cmap_black_red)
        ax.set_title(label_titles[idx])
    else:
        ax.set_title("No Label")
    ax.axis("off")

fig.suptitle("NAC Chip/Label Check", fontsize=20, fontweight="bold", y=0.995)
plt.tight_layout()



In [ ]:
# Load chips, labels
chip_datasets = [rxr.open_rasterio(f).values for f in [chip_output_filename, train_path]]
chip_single_bands = [chip.transpose(1, 2, 0)[:, :, 0] for chip in chip_datasets]
label_datasets = [np.load(f, allow_pickle=True)['data'] for f in [label_new_path, label_path]]

# Create colormap, matplotlib grid plot
cmap_black_red = ListedColormap(["black", "red"])
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot chips in first row
axes[0, 0].imshow(chip_single_bands[0], cmap='gray')
axes[0, 0].set_title("New Chip")
axes[0, 1].imshow(chip_single_bands[1], cmap='gray')
axes[0, 1].set_title("Old Chip")

# Plot labels in second row
axes[1, 0].imshow(label_datasets[0], cmap=cmap_black_red)
axes[1, 0].set_title("New Label")
axes[1, 1].imshow(label_datasets[1], cmap=cmap_black_red)
axes[1, 1].set_title("Old Label")

# Finalize plot with title, tight layout
fig.suptitle("New Chip/Label vs Original", fontsize=20, fontweight="bold", y=0.995)
plt.tight_layout()

## 8. Clean up

Close raster/xarray datasets after inspection.



In [ ]:
reference_ds.close()
new_chip_ds.close()

